# Phase-1 Full Pipeline (Colab + VSCode)

This notebook includes the full pipeline for your current goal:
- Stage A: `01~07` (core outputs)
- Stage B: `08~11` (five-metric validation/certification)


## 0) Runtime Setup (auto path detection)

If Colab cannot auto-detect, set manually:

```python
%env PHASE1_REPO_DIR=/content/drive/MyDrive/<your-repo-root>
```


In [7]:
import os
import sys
import csv
import json
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)


def _run_capture(cmd: str) -> str:
    try:
        out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return ''
    return out.strip()


def _resolve_from_env() -> tuple[Path, Path] | None:
    env_repo = os.environ.get('PHASE1_REPO_DIR', '').strip()
    if not env_repo:
        return None
    repo = Path(env_repo).expanduser().resolve()
    phase1 = repo / 'Phase-1'
    if phase1.exists() and (phase1 / '00_run_phase1.py').exists():
        return repo, phase1
    if repo.name == 'Phase-1' and (repo / '00_run_phase1.py').exists():
        return repo.parent, repo
    return None


def _resolve_from_cwd() -> tuple[Path, Path] | None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *list(cwd.parents)[:6]]
    for base in candidates:
        phase1 = base / 'Phase-1'
        if phase1.exists() and (phase1 / '00_run_phase1.py').exists():
            return base, phase1
        if base.name == 'Phase-1' and (base / '00_run_phase1.py').exists():
            return base.parent, base
    return None


def _resolve_from_marker() -> tuple[Path, Path] | None:
    marker_names = ['00_run_phase1.py', 'run_phase1_lowmem.py', '05_compute_metrics.py']
    search_roots = ['/content/drive/MyDrive', '/content/drive/Shareddrives', '/content']

    for root in search_roots:
        for marker in marker_names:
            cmd = f"find {root} -type f -name '{marker}' 2>/dev/null | head -n 1"
            hit = _run_capture(cmd)
            if not hit:
                continue
            file_path = Path(hit).resolve()
            phase1 = file_path.parent
            if (phase1 / '00_run_phase1.py').exists() and (phase1 / '05_compute_metrics.py').exists():
                return phase1.parent, phase1

    cmd = "find /content/drive -type d -name 'Phase-1' 2>/dev/null | head -n 1"
    hit = _run_capture(cmd)
    if hit:
        phase1 = Path(hit).resolve()
        if (phase1 / '00_run_phase1.py').exists():
            return phase1.parent, phase1

    return None


resolved = _resolve_from_env() or _resolve_from_cwd() or _resolve_from_marker()
if resolved is None:
    print('Could not find Phase-1 automatically.')
    print('Run this diagnostic and then set PHASE1_REPO_DIR manually:')
    print("!find /content/drive -type f -name '00_run_phase1.py' 2>/dev/null | head -n 20")
    raise FileNotFoundError('Cannot locate Phase-1. Set PHASE1_REPO_DIR to repo root and rerun.')

REPO_DIR, PHASE1_DIR = resolved
os.environ['PHASE1_REPO_DIR'] = str(REPO_DIR)

PYTHON = sys.executable
print('Python    :', PYTHON)
print('Repo dir  :', REPO_DIR)
print('Phase-1   :', PHASE1_DIR)


def run_py(script_name: str, *args: str):
    cmd = [PYTHON, str(PHASE1_DIR / script_name), *args]
    print('\n$ ' + ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(PHASE1_DIR), text=True)
    if result.returncode != 0:
        raise RuntimeError(f'{script_name} failed with exit code {result.returncode}')


Mounted at /content/drive


FileNotFoundError: Cannot locate Phase-1. Set PHASE1_REPO_DIR to repo root and rerun.

## 1) Optional dependencies (for fresh Colab only)


In [ ]:
INSTALL_DEPS = False
if INSTALL_DEPS:
    req = PHASE1_DIR / 'requirements.txt'
    if req.exists():
        cmd = [PYTHON, '-m', 'pip', 'install', '-r', str(req)]
        print('$', ' '.join(cmd))
        subprocess.run(cmd, check=False)
    else:
        print('requirements.txt not found; skipped.')
else:
    print('Dependency install skipped.')


## 2) Execution profile


In [ ]:
profile = {
    'PHASE1_DEVICE': 'auto',
    'PHASE1_CUDA_DEVICE': '0',
    'PHASE1_USE_GPU': '1',
    'PHASE1_DOMAIN_BATCH_SIZE': '256',
    'PHASE1_MAX_BATCHES': '0',
    'PHASE1_INDEX_MAX_BATCHES': '0',
    'PHASE1_TFIDF_MAX_FEATURES': '3000',
    'PHASE1_BUILD_TFIDF_MATRIX': '0',
    'PHASE1_STORE_DOC_TEXTS': '1',
    'PHASE1_DOC_TEXT_BACKEND': 'sqlite',
    'PHASE1_DOC_TEXT_INSERT_BATCH': '2000',
    'PHASE1_DOC_TEXT_CACHE_LIMIT': '2500',
    'PHASE1_ENABLE_MINHASH': '1',
    'PHASE1_QUERY_CACHE_LIMIT': '1024',
    'PHASE1_NGRAM_CACHE_LIMIT': '2048',
    'PHASE1_SEMANTIC_CANDIDATE_LIMIT': '80',
    'PHASE1_NGRAM_CANDIDATE_LIMIT': '80',
    'PHASE1_SKIP_REDUNDANCY': '0',
    'PHASE1_SKIP_PERPLEXITY': '0',
    'PHASE1_DATASETS_CONFIG': 'datasets_config.json',
    'PHASE1_IDENTITY_CONFIG': 'configs/metric_identity_v1.json',
}

for k, v in profile.items():
    os.environ[k] = v

for k in sorted(profile):
    print(f'{k}={os.environ[k]}')


## 3) Select what to run

- Full run from scratch: set all to `True`.
- If you already have outputs, keep earlier steps `False`.


In [ ]:
RUN = {
    '01': False,
    '02': False,
    '03': False,
    '04': False,
    '05': False,
    '06a': False,
    '07a': False,
    '08': True,
    '09': False,
    '10': False,
    '11': False,
    '06b': False,
    '07b': False,
}
print(RUN)


## Stage A) Core pipeline (01~07)


In [ ]:
if RUN['01']:
    run_py('01_collect_khan_academy.py')
else:
    print('[SKIP] 01_collect_khan_academy.py')


In [ ]:
if RUN['02']:
    run_py('02_collect_tiny_textbooks.py')
else:
    print('[SKIP] 02_collect_tiny_textbooks.py')


In [ ]:
if RUN['03']:
    run_py('03_extract_khan_taxonomy.py')
else:
    print('[SKIP] 03_extract_khan_taxonomy.py')


In [ ]:
if RUN['04']:
    run_py('04_build_corpus_index.py')
else:
    print('[SKIP] 04_build_corpus_index.py')


In [ ]:
if RUN['05']:
    run_py('05_compute_metrics.py')
else:
    print('[SKIP] 05_compute_metrics.py')


In [ ]:
if RUN['06a']:
    run_py('06_build_dashboard.py')
else:
    print('[SKIP] 06_build_dashboard.py (pre-cert)')


In [ ]:
if RUN['07a']:
    run_py('07_validate_phase1_outputs.py')
else:
    print('[SKIP] 07_validate_phase1_outputs.py (pre-cert)')


## Stage B) Five-metric validation/certification (08~11)


In [ ]:
required = [
    PHASE1_DIR / 'outputs' / 'khan_analysis.jsonl',
    PHASE1_DIR / 'outputs' / 'tiny_textbooks_analysis.jsonl',
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Missing required files:')
    for p in missing:
        print(' -', p)
    raise SystemExit('Need outputs from Stage A (at least through 05).')
print('Stage B preflight OK.')


In [ ]:
if RUN['08']:
    run_py('08_generate_label_templates.py')
else:
    print('[SKIP] 08_generate_label_templates.py')


### Manual labeling required after 08

Fill:
- `validation/labels/domain_labels_v1.csv`
- `validation/labels/quality_labels_v1.csv`
- `validation/labels/difficulty_sanity_v1.csv`
- `validation/labels/ood_labels_v1.csv`

For `09`, `ood_labels_v1.csv` must have labeled rows where `split=calibration`.


In [ ]:
labels_dir = PHASE1_DIR / 'validation' / 'labels'
checks = {
    'domain_labels_v1.csv': ('gold_primary', 'main_eval'),
    'quality_labels_v1.csv': ('gold_has_examples', 'main_eval'),
    'difficulty_sanity_v1.csv': ('gold_valid', 'main_eval'),
    'ood_labels_v1.csv': ('gold_in_domain', 'calibration'),
}

def non_empty(v):
    return str(v or '').strip() != ''

ok = True
for file_name, (col, split_name) in checks.items():
    path = labels_dir / file_name
    if not path.exists():
        print(f'[MISS] {file_name}')
        ok = False
        continue

    with path.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))

    split_rows = [r for r in rows if str(r.get('split', '')).strip() == split_name]
    labeled = [r for r in split_rows if non_empty(r.get(col))]
    print(f'[CHECK] {file_name}: split={split_name}, total={len(split_rows)}, labeled={len(labeled)}')

    if len(split_rows) == 0 or len(labeled) == 0:
        ok = False

if not ok:
    raise SystemExit('Labeling incomplete. Fill labels and rerun this cell.')

print('Labeling check passed.')


In [ ]:
if RUN['09']:
    run_py('09_calibrate_ood_thresholds.py')
else:
    print('[SKIP] 09_calibrate_ood_thresholds.py')


In [ ]:
if RUN['10']:
    run_py('10_score_metric_gates.py')
else:
    print('[SKIP] 10_score_metric_gates.py')


In [ ]:
if RUN['11']:
    run_py('11_certify_phase1.py')
else:
    print('[SKIP] 11_certify_phase1.py')


## Final refresh


In [ ]:
if RUN['06b']:
    run_py('06_build_dashboard.py')
else:
    print('[SKIP] 06_build_dashboard.py (post-cert)')


In [ ]:
if RUN['07b']:
    run_py('07_validate_phase1_outputs.py', '--require-gates', '--write-report', 'outputs/validation/final_validation_report.json')
else:
    print('[SKIP] 07_validate_phase1_outputs.py --require-gates (post-cert)')


## Output checkpoint


In [ ]:
targets = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'run_summary.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'ood_calibration_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_main.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_transfer.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'full_validation_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'final_validation_report.json',
    PHASE1_DIR / 'outputs' / 'dashboard.html',
]

for p in targets:
    print(f"[{'OK' if p.exists() else 'MISS'}] {p}")

manifest_path = PHASE1_DIR / 'outputs' / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('
certification_status:', manifest.get('certification_status'))
    print('certification_failed_gates:', manifest.get('certification_failed_gates'))
